In [1]:
import os
import requests
import tarfile
import pickle
import numpy as np
from tqdm import tqdm 

import ssl
ssl._create_default_https_context = ssl._create_unverified_context

import urllib3
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)


IMAGENET_URLS = {
    "devkit": "https://image-net.org/data/ILSVRC/2012/ILSVRC2012_devkit_t12.tar.gz",
    "val": "https://image-net.org/data/ILSVRC/2012/ILSVRC2012_img_val.tar",
    "train": "https://image-net.org/data/ILSVRC/2012/ILSVRC2012_img_train.tar",
}


def download_imagenet(destination="./cache/imagenet"):
    """Download ImageNet 2012."""
    os.makedirs(destination, exist_ok=True)
    paths = {}

    for key, url in IMAGENET_URLS.items():
        filename = os.path.basename(url)
        filepath = os.path.join(destination, filename)

        # Try to resume if partial file exists
        resume_header = {}
        file_mode = 'ab'  # append binary

        existing_size = os.path.getsize(filepath) if os.path.exists(filepath) else 0

        # Get total size from server
        head = requests.head(url, verify=False)
        total_size = int(head.headers.get("content-length", 0))

        if existing_size >= total_size:
            print(f"Already downloaded: {filepath}")
            paths[key] = filepath
            continue

        if existing_size > 0:
            print(f"Resuming {key} at {existing_size} bytes...")
            resume_header = {'Range': f'bytes={existing_size}-'}
        else:
            print(f"Downloading {key} from {url}")

        with requests.get(url, stream=True, headers=resume_header, verify=False) as response:
            response.raise_for_status()
            mode_desc = "Resuming" if existing_size > 0 else "Downloading"
            with open(filepath, file_mode) as f, tqdm(
                desc=f"{mode_desc} {filename}",
                total=total_size,
                initial=existing_size,
                unit='B',
                unit_scale=True,
                unit_divisor=1024,
            ) as bar:
                for chunk in response.iter_content(chunk_size=8192):
                    if chunk:
                        f.write(chunk)
                        bar.update(len(chunk))

        print(f"Saved to: {filepath}")
        paths[key] = filepath

    return paths

In [ ]:
paths = download_imagenet(destination="./data/imagenet")

Saved to: ./data/imagenet/ILSVRC2012_devkit_t12.tar.gz


In [ ]:
with tarfile.open("data/imagenet/ILSVRC2012_devkit_t12.tar.gz", "r:gz") as tar:
    tar.extractall(path="data/imagenet/devkit")


In [ ]:
with tarfile.open("data/imagenet/ILSVRC2012_img_val.tar", "r:") as tar:
    tar.extractall(path="data/imagenet/val")


In [ ]:
with tarfile.open("data/imagenet/ILSVRC2012_img_train.tar", "r:") as tar:
    tar.extractall(path="data/imagenet/train-tars")


In [7]:
import os
import tarfile

train_tar_dir = "data/imagenet/train-tars"
extracted_dir = "data/imagenet/train"

os.makedirs(extracted_dir, exist_ok=True)

for filename in os.listdir(train_tar_dir):
    if filename.endswith(".tar"):
        class_name = filename.split(".")[0]
        class_dir = os.path.join(extracted_dir, class_name)
        os.makedirs(class_dir, exist_ok=True)

        with tarfile.open(os.path.join(train_tar_dir, filename)) as tar:
            tar.extractall(path=class_dir)
